                           MULTIPLE LINEAR REGRESSION


OBJECTIVES

To use scikit-learn to implement Multiple Linear Regression
Create, train, and test a multiple linear regression model on real data to predict flood in Nigeria.

IMPORT NEEDED PACKAGES

For this lab, the following packages will be used:

NumPy
Matplotlib
Pandas
seaborn
Scikit-learn

To avoid issues importing these libraries, we execute the following cell to ensure they are available.

In [ ]:
%pip install numpy==2.2.0
%pip install pandas==2.2.3
%pip install scikit-learn==1.6.0
%pip install matplotlib==3.9.3
%pip install seaborn==0.12.3

Now we import these libraries for making the code.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns  # Added to support barplot visualization
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA SOURCE

In [3]:
url = "https://drive.google.com/uc?export=download&id=1wu3ZDXwla-FZSuuQ9Vndn_CgtUDB-NbW"

LOAD THE DATA

In [4]:
df = pd.read_csv(url)
target = 'FloodProbability'

 UNDERSTAND THE DATA

Flood.csv
I used a Kaggle Flood Prediction Dataset, Flood.csv, which contains  50000 rows and 21 columns of raw data.
The dataset includes 21 numeric variables such as
1. MonsoonIntensity, 
2. TopographyDrainage, 
3. RiverManagement,
4. Deforestation,
5. Urbanization,
6. ClimateChange,
7. DamsQuality,
8. Siltation,
9. AgriculturalPractices,
10. Encroachments,
11. IneffectiveDisasterPreparedness,
12. DrainageSystems,
13. CoastalVulnerability,
14. Landslides,
15. Watersheds,
16. DeterioratingInfrastructure,
17. PopulationScore,
18. WetlandLoss,
19. InadequatePlanning,
20. PoliticalFactors, and 
21. FloodProbability.
[Dataset source]: https://drive.google.com/file/d/1wu3ZDXwla-FZSuuQ9Vndn_CgtUDB-NbW/view?usp=sharing

This task will create a multiple linear regression model using some of these features to predict flood based on the selected features. 

EXPLORE DATA AND SELECT FEATURES

In [ ]:
df.describe()

First consider a statistical summary of the data.
The first 20 columns are whole numbers (integers like 5, 6, 12), which likely represent environmental clues like daily rainfall, river levels, or soil moisture.
The last column contains decimals (like 0.64, 0.5211), is perfect for a Regression Target (likely representing a continuous Flood Risk Index, probability, or water level).

Let's select a few features to work with that might be predictive of flood.
If there are any categorical variables it is at this stage we drop them using 

Considering the relationships among the features.

Analyzing a correlation matrix that displays the pairwise correlations between all features indicates the level of independence between them.

It also indicates how predictive each feature is of the target.

We want to eliminate any strong dependencies or correlations between features by selecting the best one from each correlated group.

In [ ]:
df.corr()

In [ ]:
axes = pd.plotting.scatter_matrix(df, alpha=0.2)
# need to rotate axis labels so we can read them
for ax in axes.flatten():
    ax.xaxis.label.set_rotation(90)
    ax.yaxis.label.set_rotation(0)
    ax.yaxis.label.set_ha('right')

plt.tight_layout()
plt.gcf().subplots_adjust(wspace=0, hspace=0)
plt.show()

CALCULATE & SAVE CORRELATIONS

In [9]:
# Use numeric_only=True to safeguard against text columns
corr_matrix = df.corr(numeric_only=True)
corr_with_target = corr_matrix[target].drop(target).abs().sort_values(ascending=False)

# Save the full correlation ranking to a CSV file
corr_with_target.to_csv('feature_correlation.csv', header=['Absolute_Correlation'])
print("✅ Saved 'feature_correlation.csv'")

✅ Saved 'feature_correlation.csv'


VISUALIZE TOP 10 FEATURES & SAVE AS PNG

In [ ]:
top_n = 10
top_features = corr_with_target.head(top_n).index.tolist()

plt.figure(figsize=(10, 6))
# CHANGED: Switched from plt.barplot to sns.barplot to avoid AttributeError
sns.barplot(x=corr_with_target.head(top_n).values, y=top_features, palette='viridis')
plt.title(f'Top {top_n} Features Correlated with {target}')
plt.xlabel('Absolute Correlation Coefficient')
plt.ylabel('Features')
plt.tight_layout()
plt.savefig('top_features_correlation.png', dpi=300) 
plt.show()

Checking the selected features

In [ ]:
df.corr()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Isolate just your top 10 feature columns
top_10_features = corr_with_target.head(10).index.tolist()

# 2. Run the scatter matrix on only those 10 columns (sets up a 10x10 layout)
axes = pd.plotting.scatter_matrix(
    df[top_10_features], 
    alpha=0.2, 
    figsize=(16, 16)  # Added to give the 100 subplots room to breathe
)

# 3. Your original rotation loop to make labels readable
for ax in axes.flatten():
    ax.xaxis.label.set_rotation(90)
    ax.yaxis.label.set_rotation(0)
    ax.yaxis.label.set_ha('right')

# 4. Your original layout adjustments
plt.tight_layout()
plt.gcf().subplots_adjust(wspace=0, hspace=0)

# 5. Optional: Save the clean 10x10 image directly to your project folder
plt.savefig('top_10_scatter_matrix.png', dpi=300, bbox_inches='tight')

plt.show()


SELECT BEST FEATURES FOR MULTIPLE REGRESSION

FIXED: Changed from 5 to 20. This dataset requires all/most features to break out of R2 = 0

In [27]:
num_features_to_select = 20
selected_features = corr_with_target.head(num_features_to_select).index.tolist()
print(f"\n Selected Top {num_features_to_select} Features:\n {selected_features}")

X = df[selected_features]
y = df[target]


 Selected Top 20 Features:
 ['DeterioratingInfrastructure', 'TopographyDrainage', 'RiverManagement', 'Watersheds', 'DamsQuality', 'PopulationScore', 'Siltation', 'IneffectiveDisasterPreparedness', 'PoliticalFactors', 'MonsoonIntensity', 'WetlandLoss', 'InadequatePlanning', 'Landslides', 'AgriculturalPractices', 'ClimateChange', 'Urbanization', 'Deforestation', 'Encroachments', 'DrainageSystems', 'CoastalVulnerability']


SPLIT, TRAIN, AND PREDICT

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

EVALUATE & SAVE METRICS TO CSV

Flatten arrays with .ravel() to guarantee clean mathematical shape matches

In [29]:
y_test_flat = y_test.values.ravel()
y_pred_flat = y_pred.ravel()

mae = mean_absolute_error(y_test_flat, y_pred_flat)
mse = mean_squared_error(y_test_flat, y_pred_flat)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_flat, y_pred_flat)

Save Evaluation Metrics

In [30]:
metrics_df = pd.DataFrame({
    'Metric': ['Mean Absolute Error (MAE)', 'Mean Squared Error (MSE)', 
               'Root Mean Squared Error (RMSE)', 'R-squared (R2)'],
    'Value': [mae, mse, rmse, r2]
})
metrics_df.to_csv('model_evaluation_metrics.csv', index=False)
print("✅ Saved 'model_evaluation_metrics.csv'")

# Save Model Coefficients (Weights)
coeff_df = pd.DataFrame({
    'Feature': ['Intercept'] + selected_features,
    'Coefficient': [model.intercept_] + list(model.coef_)
})
coeff_df.to_csv('model_coefficients.csv', index=False)
print("✅ Saved 'model_coefficients.csv'")

✅ Saved 'model_evaluation_metrics.csv'
✅ Saved 'model_coefficients.csv'


VISUALIZE ACTUAL vs PREDICTED & SAVE AS PNG

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test_flat, y_pred_flat, alpha=0.1, color='blue', edgecolors='none', s=10) # Set lower alpha & size due to massive row count
plt.plot([y_test_flat.min(), y_test_flat.max()], [y_test_flat.min(), y_test_flat.max()], 'r--', lw=2)
plt.xlabel('Actual Flood Probability')
plt.ylabel('Predicted Flood Probability')
plt.title('Actual vs Predicted Flood Probability (Multiple Linear Regression)')
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=300) 
plt.show()

print("\n--- 📊 Model Evaluation Results ---")
print(metrics_df.to_string(index=False))
print("\n🎉 All charts and data have been saved to your workspace folder!")